# Classifier Evaluation

Calls `/classify` for each fixture and measures:
- **Out-of-scope rejection recall** — blacklisted articles flagged as `out_of_scope=True`
- **False rejection rate** — in-scope articles incorrectly flagged as `out_of_scope=True`
- **Topic label F1** — precision and recall on expected topic labels
- **Scope signal accuracy** — `scope_signal` matches `expected_scope_signal` on cases where set

The classifier now runs two NLI passes:
1. **Relevance gate** — rejects non-urban-mobility articles (`out_of_scope=True`)
2. **Topic classification** — assigns topic labels from `config/topics.yaml`
3. **Scope signal** — classifies `"national"` / `"regional"` / `null` via two NLI hypotheses

The most important metric is **rejection recall = 1.0** (never miss an out-of-scope article).  
False rejections should stay below 0.10.

**Prerequisite:** NLP service running at `http://localhost:8001` (`/readyz` returns 200).

In [ ]:
import sys, time, requests
sys.path.insert(0, '.')
from _scorecard import load_fixture, print_scorecard, NLP_BASE_URL, HEADERS

cases = load_fixture('classify_cases.json')
in_scope    = [c for c in cases if not c['expected_out_of_scope']]
out_of_scope = [c for c in cases if c['expected_out_of_scope']]
scope_cases  = [c for c in cases if c.get('expected_scope_signal') is not None]
print(f'Loaded {len(cases)} cases: {len(in_scope)} in-scope, {len(out_of_scope)} blacklisted, {len(scope_cases)} with scope signal expectations')

In [ ]:
r = requests.get(f'{NLP_BASE_URL}/readyz', headers=HEADERS)
assert r.status_code == 200, 'Service not ready'

In [ ]:
results = []

for case in cases:
    t0 = time.monotonic()
    resp = requests.post(
        f'{NLP_BASE_URL}/classify',
        json={'article_id': case['article_id'], 'text': case['text']},
        headers=HEADERS
    )
    latency = time.monotonic() - t0
    assert resp.status_code == 200, f"{case['id']}: HTTP {resp.status_code}"
    data = resp.json()

    expected_oos    = case['expected_out_of_scope']
    returned_oos    = data['out_of_scope']
    oos_correct     = expected_oos == returned_oos

    expected_scope  = case.get('expected_scope_signal')   # None means "don't care"
    returned_scope  = data.get('scope_signal')
    scope_correct   = (expected_scope is None) or (returned_scope == expected_scope)

    expected_topics = set(case.get('expected_topics_include', []))
    returned_topics = set(data['topics'])
    topic_hits      = expected_topics & returned_topics
    topic_precision = len(topic_hits) / len(returned_topics) if returned_topics else (1.0 if not expected_topics else 0.0)
    topic_recall    = len(topic_hits) / len(expected_topics) if expected_topics else 1.0

    icon  = '✅' if (oos_correct and scope_correct) else '❌'
    label = 'OOS' if expected_oos else 'IN '
    results.append({
        'id': case['id'],
        'description': case['description'],
        'expected_oos': expected_oos,
        'returned_oos': returned_oos,
        'oos_correct': oos_correct,
        'expected_scope': expected_scope,
        'returned_scope': returned_scope,
        'scope_correct': scope_correct,
        'topics': list(returned_topics),
        'topic_precision': topic_precision,
        'topic_recall': topic_recall,
        'latency_s': latency,
    })

    scope_tag = f'  scope={returned_scope!r}' if returned_scope is not None else ''
    print(f"{icon} [{label}] {case['id']}: out_of_scope={returned_oos}{scope_tag}  topics={list(returned_topics)}")
    if not oos_correct:
        print(f"   ⚠ WRONG OOS: expected={expected_oos} — {case['description']}")
    if not scope_correct:
        print(f"   ⚠ SCOPE MISMATCH: expected={expected_scope!r} got={returned_scope!r}")
    print()

In [ ]:
# Rejection metrics
oos_results = [r for r in results if r['expected_oos']]
in_results  = [r for r in results if not r['expected_oos']]

rejection_recall    = sum(1 for r in oos_results if r['returned_oos']) / len(oos_results) if oos_results else 1.0
false_rejection_rate = sum(1 for r in in_results if r['returned_oos']) / len(in_results) if in_results else 0.0

# Topic metrics (in-scope only, excluding out_of_scope returns)
in_with_topics = [r for r in in_results if not r['returned_oos']]
avg_topic_prec = sum(r['topic_precision'] for r in in_with_topics) / len(in_with_topics) if in_with_topics else 0.0
avg_topic_rec  = sum(r['topic_recall']    for r in in_with_topics) / len(in_with_topics) if in_with_topics else 0.0
topic_f1 = (2 * avg_topic_prec * avg_topic_rec / (avg_topic_prec + avg_topic_rec)
            if avg_topic_prec + avg_topic_rec > 0 else 0.0)

# Scope signal metrics (only on cases with an expectation set)
scope_eval = [r for r in results if r['expected_scope'] is not None]
scope_accuracy = sum(1 for r in scope_eval if r['scope_correct']) / len(scope_eval) if scope_eval else None

avg_lat = sum(r['latency_s'] for r in results) / len(results)

metrics = {
    'Cases (total)': len(results),
    'Out-of-scope cases': len(oos_results),
    'In-scope cases': len(in_results),
    'Rejection recall (target=1.0)': rejection_recall,
    'False rejection rate (target<0.10)': false_rejection_rate,
    'Topic label precision (in-scope)': avg_topic_prec,
    'Topic label recall (in-scope)': avg_topic_rec,
    'Topic F1 (target≥0.70)': topic_f1,
}
if scope_accuracy is not None:
    metrics[f'Scope signal accuracy ({len(scope_eval)} cases)'] = scope_accuracy
metrics['Average latency (s)'] = avg_lat

print_scorecard('CLASSIFIER', metrics)

if rejection_recall < 1.0:
    missed = [r['id'] for r in oos_results if not r['returned_oos']]
    print(f'⚠ Missed OOS cases: {missed}')
    print('  → Lower relevance_threshold in config/topics.yaml')

# Scope signal distribution (in-scope articles only)
from collections import Counter
scope_dist = Counter(r['returned_scope'] for r in in_results)
print('\nScope signal distribution (in-scope articles):')
for sig, n in sorted(scope_dist.items(), key=lambda x: str(x[0])):
    marker = '⚠' if sig is None and any(r['expected_scope'] is not None for r in in_results) else ' '
    print(f'  {marker} {sig!r:<12} {n} articles')

## Tuning Guide

The key parameter is `relevance_threshold` in `config/topics.yaml`.

| Symptom | Direction | Action |
|---------|-----------|--------|
| OOS articles pass through (`rejection_recall < 1.0`) | Too permissive | **Raise** `relevance_threshold` (e.g. 0.40 → 0.50) |
| In-scope articles rejected (`false_rejection_rate > 0.10`) | Too strict | **Lower** `relevance_threshold` (e.g. 0.40 → 0.30) |
| Edge cases hard to separate | Wrong hypothesis | Rewrite `relevance_hypothesis` in `topics.yaml` to be more specific |
| Topic labels wrong / missing | Score threshold too high | Lower `score_threshold` in `topics.yaml` (default 0.5) |
| Too many noisy topic labels | Score threshold too low | Raise `score_threshold` or lower `top_k` |

After changing `topics.yaml`, restart the NLP service (taxonomy is cached on startup) and re-run this notebook.

**Binary search approach for `relevance_threshold`:**
1. Collect all scores by running the cells above with `relevance_threshold: 0.0` (pass everything through)
2. Print scores for OOS cases — the threshold should be just above the highest in-scope score
3. Start at the midpoint and iterate until `rejection_recall=1.0` and `false_rejection_rate<0.10`

In [ ]:
# Helper: show raw relevance scores to aid threshold selection
# Set relevance_threshold: 0.0 in topics.yaml and restart service, then run this cell.

from _scorecard import NLP_BASE_URL
import requests

# This re-calls classify but inspects scores (scores dict includes the relevance hypothesis score
# only if it was added as a label — for inspection, you can add it to labels temporarily).
# Alternative: call the NLI model directly for the relevance hypothesis text.
print('Score distribution analysis:')
print('  Run with relevance_threshold=0.0 in topics.yaml to see all articles pass,')
print('  then inspect topic scores to find the natural separation point.')

## Relevance Score Distribution

Calls the NLI model **directly** (bypassing the service threshold) to get raw relevance scores
for all 15 fixture cases. Use this to pick the right `relevance_threshold` — you want the value
to sit in the gap between the highest in-scope score and the lowest OOS score.

**No service restart needed** — the model is loaded here independently.

In [ ]:
from transformers import pipeline as hf_pipeline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

_RELEVANCE_HYPOTHESIS = (
    "Este artículo trata sobre movilidad urbana alternativa: ciclismo en ciudad, "
    "bicicletas, patinetes, peatonalización, infraestructura para transporte activo "
    "o política de movilidad sostenible urbana."
)
_MODEL = "Recognai/bert-base-spanish-wwm-cased-xnli"

print(f"Loading {_MODEL} ...")
nli = hf_pipeline("zero-shot-classification", model=_MODEL)
print("Done.")

In [ ]:
score_records = []
for case in cases:
    out = nli(case['text'], candidate_labels=[_RELEVANCE_HYPOTHESIS], multi_label=True)
    score = out['scores'][0]
    label = 'OOS' if case['expected_out_of_scope'] else 'IN'
    icon = '🔴' if case['expected_out_of_scope'] else '🟢'
    score_records.append({'id': case['id'], 'score': score, 'oos': case['expected_out_of_scope'], 'label': label})
    print(f"{icon} [{label}] {case['id']:10s}  relevance_score={score:.4f}  {case['description'][:60]}")

in_scores  = [r['score'] for r in score_records if not r['oos']]
oos_scores = [r['score'] for r in score_records if r['oos']]
print(f"\nIN-scope  min={min(in_scores):.3f}  max={max(in_scores):.3f}")
print(f"OOS       min={min(oos_scores):.3f}  max={max(oos_scores):.3f}")
print(f"\nSuggested threshold: {(max(oos_scores) + min(in_scores)) / 2:.3f}  (midpoint of gap)")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for i, r in enumerate(score_records):
    color = '#d62728' if r['oos'] else '#2ca02c'
    ax.scatter(r['score'], i, color=color, s=80, zorder=3)
    ax.text(r['score'] + 0.005, i, f"{r['id']} ({r['label']})", va='center', fontsize=8)

# Mark current threshold
current_threshold = 0.4
ax.axvline(current_threshold, color='orange', linestyle='--', linewidth=1.5, label=f'Current threshold ({current_threshold})')

# Shade gap
if oos_scores and in_scores:
    ax.axvspan(max(oos_scores), min(in_scores), alpha=0.15, color='blue', label='Safe zone')

in_patch  = mpatches.Patch(color='#2ca02c', label='In-scope')
oos_patch = mpatches.Patch(color='#d62728', label='Out-of-scope')
ax.legend(handles=[in_patch, oos_patch], loc='upper left')
ax.set_xlabel('Relevance score (NLI entailment)')
ax.set_yticks([])
ax.set_title('Relevance gate score distribution — fixture cases')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()